In [3]:
from pathlib import Path
import h5py, yaml, numpy as np

# Paths (adjust if you want the 2048 version)
data_dir = Path("/home/fsoto/Documents/SSLPeriodicLCs/data/alerce_data/final")
h5_path = data_dir / "dataset_windows.h5"
dict_path = data_dir / "dict_info.yaml"

# Requested feature names
selected_features = [
    "Multiband_period_12","g_W2","r_W2","g_r_max_corr_12","g_W3",
    "g_r_mean_12","g_r_max_12","GP_DRW_tau_1","g_r_mean_corr_12","IAR_phi_1",
    "Amplitude_1","ExcessVar_1","Meanvariance_1","r_W3","Std_1",
    "GP_DRW_sigma_1","GP_DRW_tau_2","PercentAmplitude_1","SF_ML_amplitude_1",
    "SPM_A_1","Gskew_1","Q31_1","Autocor_length_1","IAR_phi_2",
    "SF_ML_gamma_1","Amplitude_2",
]

# Load column names and periodic class ids
with open(dict_path, "r") as f:
    info = yaml.safe_load(f)
feat_cols = info["feat_cols"]
md_cols = info["md_cols"]
periodic_old_ids = list(info["periodic_mapping"]["periodic_old_to_new"].keys())

# Map names to column indices
feat_idx = [feat_cols.index(name) for name in selected_features]
md_idx = list(range(len(md_cols)))  # all metadata columns

# Read raw arrays from HDF5
with h5py.File(h5_path, "r") as h5f:
    feats = h5f["extracted_feat_None"][:]  # shape: (N, n_features)
    meta = h5f["metadata_feat"][:]         # shape: (N, n_metadata)
    labels = h5f["labels"][:]

# Filter to periodic classes only
periodic_mask = np.isin(labels, periodic_old_ids)
feats_periodic = feats[periodic_mask]
meta_periodic = meta[periodic_mask]

# Compute min/max
def summarize(arr, idx, names):
    return {name: (float(np.nanmin(arr[:, i])), float(np.nanmax(arr[:, i])))
            for name, i in zip(names, idx)}

feat_ranges = summarize(feats_periodic, feat_idx, selected_features)
md_ranges = summarize(meta_periodic, md_idx, [md_cols[i] for i in md_idx])

print(f"Samples total: {len(labels)}, periodic: {periodic_mask.sum()}")
print("Feature ranges (min, max) for periodic classes:")
for k, v in feat_ranges.items():
    print(f"  {k}: {v}")

print("\nMetadata ranges (min, max) for periodic classes:")
for k, v in md_ranges.items():
    print(f"  {k}: {v}")


Samples total: 259933, periodic: 181921
Feature ranges (min, max) for periodic classes:
  Multiband_period_12: (-inf, 98.54735494590798)
  g_W2: (-inf, 20.528184290185017)
  r_W2: (-inf, 22.175450067395488)
  g_r_max_corr_12: (-inf, 6.984852203413688)
  g_W3: (-inf, 21.190912154340563)
  g_r_mean_12: (-inf, 4.7317380583938125)
  g_r_max_12: (-inf, 6.537385336668702)
  GP_DRW_tau_1: (-inf, 403.4287934927351)
  g_r_mean_corr_12: (-inf, 6.044159317219361)
  IAR_phi_1: (-inf, 0.9999999848794109)
  Amplitude_1: (-inf, 4.342507863173418)
  ExcessVar_1: (-inf, 0.024713547052597658)
  Meanvariance_1: (-inf, 0.15733060279626487)
  r_W3: (-inf, 21.99845006739549)
  Std_1: (-inf, 2.5088037189753885)
  GP_DRW_sigma_1: (-inf, 178482300.96318725)
  GP_DRW_tau_2: (-inf, 403.4287934927351)
  PercentAmplitude_1: (-inf, 0.7024580426976247)
  SF_ML_amplitude_1: (-inf, 15.0)
  SPM_A_1: (-inf, 3084.321855744819)
  Gskew_1: (-inf, 6.531171553588198)
  Q31_1: (-inf, 4.72875788926887)
  Autocor_length_1: (-in